# HydroTwin GRU Model

This notebook develops Gated Recurrent Unit (GRU) models for the Turbine Guide Bearing (TGB) and Upper Guide Bearing (UGB).

Each model uses 36 historical observations containing four sensor variables to predict vibration at the next 10 minute interval.

The GRU models are developed using June to August training data and September validation data.

The November test dataset remains untouched until the final model comparison.

## 1. Import Libraries

The required libraries are imported for data processing, GRU model development, evaluation, and visualization.

In [ ]:
from pathlib import Path
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    GRU,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

## 2. Project Paths

The prepared datasets and previously fitted training scalers are loaded from the HydroTwin project directory.

In [ ]:
PROJECT_ROOT = Path("..").resolve()

PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"

TGB_DIR = PREPARED_DIR / "tgb"
UGB_DIR = PREPARED_DIR / "ugb"

GRU_DIR = PROJECT_ROOT / "models" / "gru"

SCALER_DIR = (
    PROJECT_ROOT / "models" / "lstm" / "scalers"
)

RESULTS_DIR = PROJECT_ROOT / "results"

GRU_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("GRU model directory:", GRU_DIR)
print("Existing scaler directory:", SCALER_DIR)

## 3. Load TGB and UGB Datasets

The same prepared sequences used in the Linear Regression and LSTM experiments are loaded to ensure a fair model comparison.

In [ ]:
X_train_tgb = np.load(TGB_DIR / "X_train.npy")
y_train_tgb = np.load(TGB_DIR / "y_train.npy")

X_val_tgb = np.load(TGB_DIR / "X_val.npy")
y_val_tgb = np.load(TGB_DIR / "y_val.npy")

print("TGB Training:", X_train_tgb.shape)
print("TGB Validation:", X_val_tgb.shape)

In [ ]:
X_train_ugb = np.load(UGB_DIR / "X_train.npy")
y_train_ugb = np.load(UGB_DIR / "y_train.npy")

X_val_ugb = np.load(UGB_DIR / "X_val.npy")
y_val_ugb = np.load(UGB_DIR / "y_val.npy")

print("UGB Training:", X_train_ugb.shape)
print("UGB Validation:", X_val_ugb.shape)

## 4. Load Training Scalers

The previously fitted LSTM feature and target scalers are reused for GRU.

This ensures both recurrent neural network models use the same scaling method without introducing information from the validation dataset.

In [ ]:
tgb_feature_scaler = joblib.load(
    SCALER_DIR / "tgb_feature_scaler.pkl"
)

tgb_target_scaler = joblib.load(
    SCALER_DIR / "tgb_target_scaler.pkl"
)

ugb_feature_scaler = joblib.load(
    SCALER_DIR / "ugb_feature_scaler.pkl"
)

ugb_target_scaler = joblib.load(
    SCALER_DIR / "ugb_target_scaler.pkl"
)

print("All four scalers loaded successfully.")

In [ ]:
X_train_tgb_scaled = (
    tgb_feature_scaler
    .transform(X_train_tgb.reshape(-1, 4))
    .reshape(X_train_tgb.shape)
)

X_val_tgb_scaled = (
    tgb_feature_scaler
    .transform(X_val_tgb.reshape(-1, 4))
    .reshape(X_val_tgb.shape)
)

y_train_tgb_scaled = (
    tgb_target_scaler
    .transform(y_train_tgb.reshape(-1, 1))
    .ravel()
)

y_val_tgb_scaled = (
    tgb_target_scaler
    .transform(y_val_tgb.reshape(-1, 1))
    .ravel()
)

print("TGB scaled training:", X_train_tgb_scaled.shape)
print("TGB scaled validation:", X_val_tgb_scaled.shape)

In [ ]:
X_train_ugb_scaled = (
    ugb_feature_scaler
    .transform(X_train_ugb.reshape(-1, 4))
    .reshape(X_train_ugb.shape)
)

X_val_ugb_scaled = (
    ugb_feature_scaler
    .transform(X_val_ugb.reshape(-1, 4))
    .reshape(X_val_ugb.shape)
)

y_train_ugb_scaled = (
    ugb_target_scaler
    .transform(y_train_ugb.reshape(-1, 1))
    .ravel()
)

y_val_ugb_scaled = (
    ugb_target_scaler
    .transform(y_val_ugb.reshape(-1, 1))
    .ravel()
)

print("UGB scaled training:", X_train_ugb_scaled.shape)
print("UGB scaled validation:", X_val_ugb_scaled.shape)

In [ ]:
print("TGB")
print("Training:", X_train_tgb_scaled.shape)
print("Validation:", X_val_tgb_scaled.shape)

print("\nUGB")
print("Training:", X_train_ugb_scaled.shape)
print("Validation:", X_val_ugb_scaled.shape)

print("\nAll values finite:",
    all(
        np.isfinite(arr).all()
        for arr in [
            X_train_tgb_scaled,
            X_val_tgb_scaled,
            X_train_ugb_scaled,
            X_val_ugb_scaled,
            y_train_tgb_scaled,
            y_val_tgb_scaled,
            y_train_ugb_scaled,
            y_val_ugb_scaled
        ]
    )
)

## 5. TGB GRU Model Development

A Gated Recurrent Unit (GRU) model is developed for Turbine Guide Bearing vibration forecasting.

The model receives 36 historical observations containing four input variables and predicts vibration at the next 10 minute interval.

A single GRU layer with 64 units is used, followed by a dropout layer and a dense output layer.

In [ ]:
tgb_gru_model = Sequential([
    Input(shape=(36, 4)),

    GRU(64),

    Dropout(0.2),

    Dense(1)
])

tgb_gru_model.compile(
    optimizer="adam",
    loss="mse"
)

tgb_gru_model.summary()

In [ ]:
tgb_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

tgb_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

## 6. TGB GRU Training

The GRU model is trained using the June to August dataset.

September is used for validation and early stopping. A maximum of 100 epochs and a batch size of 32 are applied, consistent with the LSTM experiment.

In [ ]:
tgb_gru_history = tgb_gru_model.fit(
    X_train_tgb_scaled,
    y_train_tgb_scaled,

    validation_data=(
        X_val_tgb_scaled,
        y_val_tgb_scaled
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        tgb_early_stopping,
        tgb_reduce_lr
    ],

    verbose=1
)

## 7. TGB Training and Validation Loss

Training and validation losses are visualized to assess model convergence and potential overfitting.

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    tgb_gru_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    tgb_gru_history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.title("TGB GRU Training History")

plt.legend()
plt.tight_layout()
plt.show()

## 8. TGB GRU Validation Predictions

The trained GRU model generates vibration predictions for the September validation dataset.

Predictions are converted back to the original vibration scale using the previously fitted TGB target scaler.

In [ ]:
tgb_gru_pred_scaled = tgb_gru_model.predict(
    X_val_tgb_scaled,
    verbose=0
)

tgb_gru_pred = (
    tgb_target_scaler
    .inverse_transform(
        tgb_gru_pred_scaled.reshape(-1, 1)
    )
    .ravel()
)

print("Prediction shape:", tgb_gru_pred.shape)
print("Actual shape:", y_val_tgb.shape)

## 9. TGB GRU Validation Performance

The GRU model is evaluated using RMSE, MAE, MAPE, and R squared.

These metrics are calculated on the original vibration scale to support comparison with Linear Regression and LSTM.

In [ ]:
tgb_gru_rmse = np.sqrt(
    mean_squared_error(
        y_val_tgb,
        tgb_gru_pred
    )
)

tgb_gru_mae = mean_absolute_error(
    y_val_tgb,
    tgb_gru_pred
)

tgb_gru_mape = (
    mean_absolute_percentage_error(
        y_val_tgb,
        tgb_gru_pred
    ) * 100
)

tgb_gru_r2 = r2_score(
    y_val_tgb,
    tgb_gru_pred
)

print(f"TGB GRU Validation RMSE: {tgb_gru_rmse:.4f}")
print(f"TGB GRU Validation MAE: {tgb_gru_mae:.4f}")
print(f"TGB GRU Validation MAPE: {tgb_gru_mape:.2f}%")
print(f"TGB GRU Validation R²: {tgb_gru_r2:.4f}")

## 10. TGB Validation Model Comparison

The GRU validation performance is compared with the previously developed Linear Regression and LSTM models.

RMSE is used as the primary comparison metric.

In [ ]:
tgb_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "LSTM",
        "GRU"
    ],

    "RMSE": [
        0.7504,
        0.7044,
        tgb_gru_rmse
    ],

    "MAE": [
        0.5126,
        0.4995,
        tgb_gru_mae
    ],

    "MAPE": [
        10.49,
        10.46,
        tgb_gru_mape
    ],

    "R2": [
        0.7094,
        0.7439,
        tgb_gru_r2
    ]
})

tgb_comparison.sort_values(
    "RMSE"
).round(4)

In [ ]:
plot_points = 500

plt.figure(figsize=(14, 5))

plt.plot(
    y_val_tgb[-plot_points:],
    label="Actual TGB Vibration"
)

plt.plot(
    tgb_gru_pred[-plot_points:],
    label="GRU Prediction"
)

plt.xlabel("Validation Observation")
plt.ylabel("TGB Vibration")

plt.title(
    "TGB GRU Validation: Actual vs Predicted"
)

plt.legend()
plt.tight_layout()
plt.show()

## 11. Save TGB GRU Model

The trained GRU model is saved for subsequent final test evaluation and integration into the HydroTwin digital twin prototype.

In [ ]:
tgb_gru_path = GRU_DIR / "tgb_gru.keras"

tgb_gru_model.save(
    tgb_gru_path
)

print(
    "TGB GRU model saved:",
    tgb_gru_path.exists()
)

## 12. UGB GRU Model Development

A Gated Recurrent Unit (GRU) model is developed to forecast Upper Guide Bearing vibration.

The UGB model uses the same architecture as the TGB model to maintain methodological consistency.

The model receives 36 historical observations containing four sensor variables and predicts the next 10-minute vibration reading.

In [ ]:
ugb_gru_model = Sequential([
    Input(shape=(36, 4)),

    GRU(64),

    Dropout(0.2),

    Dense(1)
])

ugb_gru_model.compile(
    optimizer="adam",
    loss="mse"
)

ugb_gru_model.summary()

In [ ]:
ugb_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

ugb_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

## 13. UGB GRU Training

The UGB GRU model is trained using June to August data and validated using September data.

The same hyperparameters and training strategy as the TGB GRU model are applied, including early stopping and learning rate reduction.

In [ ]:
ugb_gru_history = ugb_gru_model.fit(
    X_train_ugb_scaled,
    y_train_ugb_scaled,

    validation_data=(
        X_val_ugb_scaled,
        y_val_ugb_scaled
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        ugb_early_stopping,
        ugb_reduce_lr
    ],

    verbose=1
)

## 14. UGB Training and Validation Loss

Training and validation losses are examined to assess GRU convergence and identify potential overfitting.

In [ ]:
ugb_gru_epochs = len(
    ugb_gru_history.history["loss"]
)

print(
    "UGB GRU epochs completed:",
    ugb_gru_epochs
)

print(
    "Best validation loss:",
    min(ugb_gru_history.history["val_loss"])
)

plt.figure(figsize=(10, 5))

plt.plot(
    ugb_gru_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    ugb_gru_history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("UGB GRU Training History")

plt.legend()
plt.tight_layout()
plt.show()

## 15. UGB GRU Validation Predictions

The trained model generates predictions for the September UGB validation dataset.

Predictions are inverse-transformed into the original vibration scale before evaluation.

In [ ]:
ugb_gru_pred_scaled = ugb_gru_model.predict(
    X_val_ugb_scaled,
    verbose=0
)

ugb_gru_pred = (
    ugb_target_scaler
    .inverse_transform(
        ugb_gru_pred_scaled.reshape(-1, 1)
    )
    .ravel()
)

print("Prediction shape:", ugb_gru_pred.shape)
print("Actual shape:", y_val_ugb.shape)

## 16. UGB GRU Validation Performance

The UGB GRU model is evaluated using RMSE, MAE, MAPE, and R squared.

The performance is compared with Linear Regression and LSTM using the same September validation observations.

In [ ]:
ugb_gru_rmse = np.sqrt(
    mean_squared_error(
        y_val_ugb,
        ugb_gru_pred
    )
)

ugb_gru_mae = mean_absolute_error(
    y_val_ugb,
    ugb_gru_pred
)

ugb_gru_mape = (
    mean_absolute_percentage_error(
        y_val_ugb,
        ugb_gru_pred
    ) * 100
)

ugb_gru_r2 = r2_score(
    y_val_ugb,
    ugb_gru_pred
)

print(f"UGB GRU Validation RMSE: {ugb_gru_rmse:.4f}")
print(f"UGB GRU Validation MAE: {ugb_gru_mae:.4f}")
print(f"UGB GRU Validation MAPE: {ugb_gru_mape:.2f}%")
print(f"UGB GRU Validation R²: {ugb_gru_r2:.4f}")

## 17. Save UGB GRU Model

The trained UGB GRU model is saved for final evaluation and integration into the HydroTwin digital twin prototype.

The model configuration is finalized before evaluation on the unseen November dataset.

In [ ]:
ugb_gru_path = GRU_DIR / "ugb_gru.keras"

ugb_gru_model.save(ugb_gru_path)

print("UGB GRU model saved:", ugb_gru_path.exists())

print(
    "TGB GRU model saved:",
    (GRU_DIR / "tgb_gru.keras").exists()
)

## 18. GRU Validation Results

The validation metrics for both bearing components are recorded for the final comparison of Linear Regression, LSTM, and GRU.

In [ ]:
tgb_gru_epochs = 28
ugb_gru_epochs = 17

print("TGB GRU epochs:", tgb_gru_epochs)
print("UGB GRU epochs:", ugb_gru_epochs)

In [ ]:
gru_validation_results = pd.DataFrame([
    {
        "Component": "TGB",
        "Model": "GRU",
        "RMSE": tgb_gru_rmse,
        "MAE": tgb_gru_mae,
        "MAPE": tgb_gru_mape,
        "R2": tgb_gru_r2,
        "Epochs": tgb_gru_epochs
    },
    {
        "Component": "UGB",
        "Model": "GRU",
        "RMSE": ugb_gru_rmse,
        "MAE": ugb_gru_mae,
        "MAPE": ugb_gru_mape,
        "R2": ugb_gru_r2,
        "Epochs": ugb_gru_epochs
    }
])

gru_validation_results.round(4)

## 19. Save GRU Validation Results

The validation results for both TGB and UGB GRU models are saved as a CSV file.

These results will be used in the final comparison of the three selected forecasting models.

In [ ]:
gru_results_path = (
    RESULTS_DIR / "gru_validation_results.csv"
)

gru_validation_results.to_csv(
    gru_results_path,
    index=False
)

print("GRU validation results saved successfully.")

print("File path:", gru_results_path)

print("File exists:", gru_results_path.exists())

## 20. Verify GRU Model Files and Results

The saved GRU models and validation results are verified to ensure that both bearing models are ready for the final testing and comparison stage.

In [ ]:
files_to_verify = {
    "TGB GRU Model":
        GRU_DIR / "tgb_gru.keras",

    "UGB GRU Model":
        GRU_DIR / "ugb_gru.keras",

    "GRU Validation Results":
        RESULTS_DIR / "gru_validation_results.csv",

    "Linear Regression Results":
        RESULTS_DIR / "linear_regression_validation_results.csv",

    "LSTM Results":
        RESULTS_DIR / "lstm_validation_results.csv"
}

for name, path in files_to_verify.items():
    print(f"{name}: {path.exists()}")

In [ ]:
saved_gru_results = pd.read_csv(
    RESULTS_DIR / "gru_validation_results.csv"
)

print("Final GRU Validation Results")

display(
    saved_gru_results.round(4)
)